# API

In [14]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"
)

response = client.chat.completions.create(
    model="default",
    messages=[{"role": "user", "content": "Say hello in 5 words."}]
)

print(response.choices[0].message.content)

"Hello, how are you?"


# Knowledge Base

In [15]:
documents = [
    "The Aurora Bike Company was founded in 2015 in Portland, Oregon.",
    "Aurora bikes are known for their lightweight aluminum frames.",
    "The best-selling model is the Aurora Sprint, released in 2019.",
    "Aurora offers a 2-year warranty on all frames.",
    "The company's headquarters has a small museum showing bike history.",
    "Aurora's electric bike line launched in 2022, called Aurora Volt.",
    "Customer support is available Monday to Friday, 9am-5pm PST.",
    "Aurora bikes are sold in over 30 countries worldwide.",
    "The Aurora Sprint weighs only 9.5 kg.",
    "Aurora's return policy allows returns within 30 days of purchase."
]

print(f"Loaded {len(documents)} documents")

Loaded 10 documents


# Embeddings

In [16]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load a small, fast, free embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for our documents
embeddings = embed_model.encode(documents).tolist()

print(f"Created {len(embeddings)} embeddings, each of size {len(embeddings[0])}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Created 10 embeddings, each of size 384


In [19]:
len(embeddings)

10

# Vector DB

In [20]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="aurora_bikes")

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print("Documents stored in vector DB")

Documents stored in vector DB


# Retrieve 

In [21]:
def retrieve(query, n_results=3):
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    return results['documents'][0]

# quick test
test_results = retrieve("How much does the Aurora Sprint weigh?")
for r in test_results:
    print("-", r)

- The Aurora Sprint weighs only 9.5 kg.
- The best-selling model is the Aurora Sprint, released in 2019.
- Aurora bikes are known for their lightweight aluminum frames.


# Simple RAG

In [22]:
def rag_answer(query):
    # 1. Retrieve relevant chunks
    context_docs = retrieve(query)
    context = "\n".join(context_docs)

    # 2. Build prompt with context
    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know."

Context:
{context}

Question: {query}
Answer:"""

    # 3. Ask the LLM
    response = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# test it
print(rag_answer("How much does the Aurora Sprint weigh?"))

9.5 kg


In [23]:
print(rag_answer("What color is the Aurora Sprint?"))

I don't know.
